# DeepACO Reconstruct Demo (Speed Limit Only)

本 Notebook 只使用 `DeepACO_reconstruct` 目录下的重构代码，演示：
1. **不训练**：随机初始化 heuristic 网络，10 只蚂蚁、3 轮搜索，输出 optimal path；
2. **训练**：分为 Reinforce 与 GRPO 两个 block。

并且统一满足你的约束：
- policy type 固定为 `speed_limit`；
- 原始道路网络边的 `length/free_time` 比例一致（等价于统一原始速度）；
- speed limit 严格小于原始速度。


In [ ]:
import random
from pathlib import Path

import numpy as np
import torch
import networkx as nx

ROOT = Path.cwd()
if (ROOT / 'DeepACO_reconstruct').exists():
    import sys
    sys.path.insert(0, str(ROOT / 'DeepACO_reconstruct'))

from build_graph_data import build_cordon_graph_data
from model import PolicyAwareHeuristicNet
from deepaco import DeepACOAgent
from cordon_environment import CordonEnv
from train import build_reward_fn, reinforce_train, grpo_train, split_dataset

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)
device = 'cpu'
print('device =', device)


In [ ]:
def build_speed_limit_instance(
    n_rows: int = 3,
    n_cols: int = 3,
    seed: int = 0,
    base_speed: float = 60.0,
    speed_limit_inside: float = 35.0,
    speed_limit_outside: float = 45.0,
    virtual_node: int = -1,
):
    assert speed_limit_inside < base_speed
    assert speed_limit_outside < base_speed

    rng = np.random.default_rng(seed)
    G = nx.grid_2d_graph(n_rows, n_cols)
    G = nx.convert_node_labels_to_integers(G, ordering='sorted')

    links = []
    for u, v in G.edges():
        length = float(rng.uniform(1.0, 3.0))
        free_time = float(length / base_speed)  # length/free_time == base_speed (constant)
        capacity = float(rng.uniform(120.0, 350.0))

        links.append((int(u), int(v), length, free_time, capacity))
        links.append((int(v), int(u), length, free_time, capacity))

    nodes = list(G.nodes())
    all_pairs = [(o, d) for o in nodes for d in nodes if o != d]
    k = max(1, len(all_pairs) // 4)
    choose = rng.choice(len(all_pairs), size=k, replace=False)
    od_demand = {}
    for idx in choose:
        o, d = all_pairs[int(idx)]
        od_demand[(int(o), int(d))] = float(rng.uniform(40.0, 180.0))

    pack = build_cordon_graph_data(
        links=links,
        od_demand=od_demand,
        policy_type='speed_limit',
        virtual_node=virtual_node,
    )

    instance = {
        'policy_type': 'speed_limit',
        'policy_value': {'inside': float(speed_limit_inside), 'outside': float(speed_limit_outside)},
        'virtual_node': int(virtual_node),
        'links': pack['links'],
        'od_demand': pack['od_demand'],
        'road_graph': pack['road_graph'],
        'pyg_data': pack['model_data'],
        'model_data': pack['model_data'],
        'node2idx': pack['node2idx'],
        'idx2node': pack['idx2node'],
    }
    return instance

demo_inst = build_speed_limit_instance(seed=123)
ratios = [l / t for (_, _, l, t, _) in demo_inst['links'] if t > 0]
print('ratio min/max (length/free_time):', min(ratios), max(ratios))
print('speed limit policy:', demo_inst['policy_value'])


## (1) 不训练：随机初始化 heuristic + ACO 推理
10 只蚂蚁，3 轮，输出 optimal path。


In [ ]:
inst = build_speed_limit_instance(seed=100)
data = inst['pyg_data'].to(device)

# 随机初始化（不训练）
model = PolicyAwareHeuristicNet(hidden_dim=64).to(device)
model.eval()
with torch.no_grad():
    heu_mat = model(data)

reward_fn = build_reward_fn(inst)

def env_factory():
    return CordonEnv(
        road_graph=inst['road_graph'],
        max_steps=inst['road_graph'].number_of_nodes() + 2,
        virtual_node=inst.get('virtual_node', -1),
    )

agent = DeepACOAgent(
    node2idx=data.node2idx,
    n_ants=10,
    alpha=1.0,
    beta=1.0,
    decay=0.9,
    max_steps=inst['road_graph'].number_of_nodes() + 2,
    virtual_node=inst.get('virtual_node', -1),
    device=device,
)

out = agent.run(
    env_factory=env_factory,
    heu_mat=heu_mat,
    reward_fn=reward_fn,
    n_rounds=3,
    inference=False,
    update_pheromone=True,
    terminal_reward_only=True,
    policy_kind='speed_limit',
    policy_value=inst['policy_value'],
)

print('best_round =', out['best_round'])
print('best_reward =', out['best_reward'])
print('optimal_path =', out['best_path'])


## (2) 训练 block A：Reinforce
这里演示一个可跑通的最小训练（epoch 较小，便于快速试跑）。


In [ ]:
dataset = [build_speed_limit_instance(seed=1000 + i) for i in range(12)]
train_set, val_set = split_dataset(dataset, val_ratio=0.25, seed=42)

reinforce_model, reinforce_hist = reinforce_train(
    train_set=train_set,
    val_set=val_set,
    device=device,
    epochs=2,
    lr=1e-3,
    hidden_dim=64,
    n_ants=10,
    checkpoint_path='reinforce_speedlimit_best.pt',
)

print('reinforce last record:', reinforce_hist[-1] if reinforce_hist else None)


## (2) 训练 block B：GRPO
同样使用 speed_limit-only 数据。


In [ ]:
grpo_model, grpo_hist = grpo_train(
    train_set=train_set,
    val_set=val_set,
    device=device,
    epochs=2,
    lr=1e-3,
    hidden_dim=64,
    n_ants=10,
    grpo_updates=2,
    checkpoint_path='grpo_speedlimit_best.pt',
)

print('grpo last record:', grpo_hist[-1] if grpo_hist else None)
